# EX-02 Tide · Python + numpy

**A film · renders as it stands · no 3-D package at all**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dcuartielles/wisdome-360/blob/main/docs/examples/ex02-tide.ipynb)

Ribbons of light drift around the dome, crossing each other as they go. Twenty seconds, 600 frames, about 130 lines of numpy. Nothing to install beyond `numpy` and `Pillow`.

![A frame of Tide as a dome master](https://dcuartielles.github.io/wisdome-360/examples/images/ex02-tide.jpg)

| File | What it is |
|---|---|
| This notebook | The piece, cell by cell. Run it in Colab or in Jupyter |
| [`tide.py`](https://dcuartielles.github.io/wisdome-360/examples/files/EX-02-python-numpy/tide.py) | The same piece as one script: `python3 tide.py` renders the frames and encodes the `.mp4` |
| [Rendered result (.mp4)](https://github.com/dcuartielles/wisdome-360/releases/download/examples-media/EX-02_tide_2048x2048_30fps.mp4) | 2048², 20 s, on GitHub Releases. Open it in the [Dome Simulator](https://dcuartielles.github.io/wisdome-360/tools/dome-simulator.html) |

## The idea worth stealing

> Author the picture in **(azimuth, elevation)** as the audience sees it (azimuth 0 = the front of the room, elevation 0 = eye level) and convert to a fisheye dome master at the very last step.

Everything a dome asks of you becomes simple in those coordinates:

| You want to… | In (azimuth, elevation) it is |
|---|---|
| put the subject where people can see it | a box **in front**: −60° ≤ azimuth ≤ 60°, 15° ≤ elevation ≤ 60°, aimed at the sweet spot (0°, 40°) |
| put it at the front of the room | azimuth 0 |
| sweep it around the audience | add to azimuth over time |
| avoid roll | never rotate the image |

The conversion at the end, `to_fisheye()`, is the only place that knows the dome is **tilted 26°**, raised at the back.

In [ ]:
%pip install -q numpy pillow

In [ ]:
import math, os, subprocess
import numpy as np
from PIL import Image
from IPython.display import display

## Settings

Design at `RES = 1024`: a 2048² frame takes four times as long. Switch to 2048 for the final render.

In [ ]:
RES     = 1024          # 2048 for delivery
FRAMES  = 600           # 600 at 30 fps = 20 s
FPS     = 30
OUTDIR  = "frames"
TILT    = 26.0          # dome tilt in degrees, back raised (Wisdome Malmö)

# The (azimuth, elevation) canvas, in the AUDIENCE's angles. Width covers
# 360°; height covers -30..90°, because at the front of a tilted dome the
# picture reaches below eye level. W = 3*H keeps one pixel the same angle in
# both directions.
EL_MIN = -30.0
H_EQ = 1024
W_EQ = H_EQ * 3

AZ = np.linspace(0, 360, W_EQ, endpoint=False)[None, :]       # azimuth of each column
EL = np.linspace(90, EL_MIN, H_EQ, endpoint=False)[:, None]   # elevation of each row

## The pieces of the picture

A **ribbon** is a closed curve at a wobbling elevation. Read the note on the drift: it is the bug everybody writes first.

In [ ]:
def ribbon(t, elev0, amp, lobes, speed, thickness, colour, glow=1.0,
           rise=0.0, rise_rate=0.0):
    """One ribbon of light.

    elev0     centre elevation above eye level, in degrees
    amp       how far it wanders above and below elev0, in degrees
    lobes     how many times it wobbles around the full 360°
    speed     degrees of AZIMUTH the ribbon drifts per second
    thickness vertical half-width, in degrees
    rise      how far the centre elevation breathes, in degrees
    rise_rate how fast it breathes, in radians per second

    The drift is subtracted from AZIMUTH *before* multiplying by `lobes`:
        phase = radians((AZ - speed * t) * lobes)        # correct
        phase = radians(AZ * lobes + speed * t)          # WRONG: 1/lobes of the motion
    """
    drift  = speed * t
    phase  = np.radians((AZ - drift) * lobes)
    centre = elev0 + amp * np.sin(phase)
    if rise:
        centre = centre + rise * math.sin(t * rise_rate + elev0 * 0.11)
    d = np.abs(EL - centre) / thickness
    # a smooth falloff: a hard edge aliases badly once stretched across the dome
    a = np.exp(-(d ** 2) * 2.2) * glow
    return a[..., None] * np.array(colour)[None, None, :]


def horizon_glow(t, height, colour, strength):
    """A soft wash rising from eye level. Grounds the image without lighting up the dome."""
    a = np.clip(1.0 - EL / height, 0, 1) ** 2.2 * np.clip((EL + 12.0) / 12.0, 0, 1)
    a = a * (0.62 + 0.38 * math.sin(t * 0.62))
    return (a * strength)[..., None] * np.array(colour)[None, None, :]


def stars(seed=3, n=900):
    """A static point field. Texture at the zenith, not content.
    arcsin() spreads the stars evenly ON THE DOME; uniform elevation would clump them at the top."""
    rng = np.random.default_rng(seed)
    img = np.zeros((H_EQ, W_EQ, 3), np.float32)
    el = np.degrees(np.arcsin(rng.random(n)))
    az = rng.random(n) * 360.0
    ys = ((90 - el) / (90 - EL_MIN) * H_EQ).astype(int).clip(0, H_EQ - 1)
    xs = ((az / 360) * W_EQ).astype(int).clip(0, W_EQ - 1)
    b = rng.random(n) ** 2.5 * 0.55
    img[ys, xs] = np.stack([b * 0.75, b * 0.85, b], 1)
    return img

## The one conversion: `to_fisheye()`

In a dome master the centre of the circle is the **dome's** zenith, the edge is the dome's rim, radius is linear in angle, and the front of the room is at the bottom. Because the dome is tilted, its zenith is 64° up in front of the audience. So each master pixel becomes a dome direction, is tilted into the room, and is looked up on the canvas by its room azimuth and elevation.

In [ ]:
def to_fisheye(eq, n):
    """(azimuth, elevation) canvas -> equidistant dome master, n x n."""
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    c = (n - 1) / 2.0
    dx, dy = xx - c, yy - c
    r = np.sqrt(dx * dx + dy * dy) / (n / 2.0)
    inside = r <= 1.0
    th = np.radians(90.0) * np.clip(r, 0, 1)        # angle from the dome's zenith
    a = np.arctan2(dx, dy)                          # 0 = bottom = front
    lx, ly, lz = np.sin(th) * np.sin(a), np.sin(th) * np.cos(a), np.cos(th)
    ct, st = math.cos(math.radians(TILT)), math.sin(math.radians(TILT))
    x, y, z = lx, ly * ct + lz * st, -ly * st + lz * ct   # dome -> room
    elev = np.degrees(np.arcsin(np.clip(z, -1, 1)))
    az = (np.degrees(np.arctan2(x, y)) + 360.0) % 360.0
    sx = az / 360.0 * (W_EQ - 1)
    sy = np.clip((90.0 - elev) / (90.0 - EL_MIN), 0, 1) * (H_EQ - 1)
    x0 = np.floor(sx).astype(np.int32); y0 = np.floor(sy).astype(np.int32)
    x1 = (x0 + 1) % W_EQ;               y1 = np.clip(y0 + 1, 0, H_EQ - 1)
    fx = (sx - x0)[..., None];          fy = (sy - y0)[..., None]
    out = (eq[y0, x0] * (1 - fx) * (1 - fy) + eq[y0, x1] * fx * (1 - fy)
         + eq[y1, x0] * (1 - fx) * fy      + eq[y1, x1] * fx * fy)
    out[~inside] = 0
    return out

## The piece: `BANDS`

Each row is one ribbon. **This is the cell to change first.**

- `elev0`: where it sits. Stay between 15 and 60 for anything that matters.
- `lobes`: wobbles around the circle. **Integers only**: `f(0)` must equal `f(360)`, or you get a hard seam down the front of the dome.
- `speed`: degrees of **azimuth** per second. Mixing positive and negative keeps the ribbons crossing instead of moving as one block.
- `rise`, `rate`: how far and how fast the ribbon's centre breathes. This is what makes ribbons pass through each other.

In [ ]:
BANDS = [
    #   elev0  amp lobes speed  thick  colour (R, G, B)       glow  rise  rate
    (46.0,  7.0,  3,   6.0,  2.6,  (1.00, 0.34, 0.10),  0.95,  5.0, 0.30),
    (34.0,  9.0,  5,  -4.5,  2.0,  (1.00, 0.62, 0.18),  0.80,  4.0, 0.22),
    (26.0,  5.0,  2,   8.0,  3.4,  (0.20, 0.45, 1.00),  0.85,  6.0, 0.17),
    (19.0,  4.0,  7,  -3.0,  1.5,  (0.55, 0.85, 1.00),  0.55,  3.0, 0.41),
    (55.0,  6.0,  4,   2.5,  1.8,  (0.85, 0.30, 0.65),  0.45,  4.5, 0.26),
]

STARS = stars()

def frame(t):
    eq = np.zeros((H_EQ, W_EQ, 3), np.float32)
    eq += STARS
    eq += horizon_glow(t, 22.0, (0.30, 0.16, 0.45), 0.34)
    for (e0, amp, lob, spd, th, col, g, rise, rate) in BANDS:
        pulse = 0.68 + 0.32 * math.sin(t * 0.75 + e0 * 0.09)          # each on its own clock
        thick = th * (0.75 + 0.45 * (0.5 + 0.5 * math.sin(t * 0.43 + lob)))
        eq += ribbon(t, e0, amp, lob, spd, thick, col, g * pulse, rise=rise, rise_rate=rate)
    return eq


def master(t, n=RES):
    img = np.clip(to_fisheye(frame(t), n), 0, 1) ** (1 / 1.9)       # gentle gamma
    return Image.fromarray((img * 255).astype(np.uint8))

## Look at one frame

Front of the room at the bottom. The warm ribbons run through the comfortable zone, just below the centre of the circle.

In [ ]:
display(master(5.0, 768))

## Check that it moves

If a piece looks static, don't assume it's a taste problem: render two frames and subtract them. Bright areas moved; a near-black difference means almost nothing changed.

In [ ]:
a = np.asarray(master(0.0, 512), np.float32)
b = np.asarray(master(10.0, 512), np.float32)
diff = np.clip(np.abs(a - b) * 2, 0, 255).astype(np.uint8)
print("mean change per pixel:", round(float(np.abs(a - b).mean()), 1))
display(Image.fromarray(diff))

## Render the whole piece

600 frames. At 1024² this takes a few minutes; at 2048² about four times as long. The encode line is the one in the [delivery spec](https://dcuartielles.github.io/wisdome-360/reference/delivery-spec/#video). It needs `ffmpeg`, which Colab already has.

Set `RENDER = True` to run it.

In [ ]:
RENDER = False

if RENDER:
    os.makedirs(OUTDIR, exist_ok=True)
    for i in range(FRAMES):
        master(i / FPS).save(f"{OUTDIR}/frame_{i:04d}.png")
        if i % 50 == 0:
            print(f"frame {i + 1}/{FRAMES}")
    mp4 = f"tide_{RES}x{RES}_30fps.mp4"
    subprocess.run(["ffmpeg", "-y", "-framerate", str(FPS), "-i", f"{OUTDIR}/frame_%04d.png",
                    "-c:v", "libx264", "-profile:v", "high", "-pix_fmt", "yuv420p",
                    "-b:v", "60M", "-maxrate", "60M", "-bufsize", "120M", "-r", str(FPS), mp4],
                   check=True)
    print("done:", mp4)
    try:
        from google.colab import files      # in Colab: download the result
        files.download(mp4)
    except ImportError:
        pass

## Try changing

- Add a row to `BANDS`, delete one, change a colour. Then look at it in the [Dome Simulator](https://dcuartielles.github.io/wisdome-360/tools/dome-simulator.html).
- Set every `speed` to the same value. Now the whole picture turns as one rigid thing, which is what makes people feel sick (vection). Independent ribbons moving through a still frame don't. Six degrees per second is safe here only because they move independently.
- Give one ribbon `lobes = 2.5` and look at the front of the dome: that is the azimuth seam.

## What usually breaks

| Symptom | Cause |
|---|---|
| The piece looks frozen, with a faint shimmer | The drift added to the phase after multiplying by `lobes`: you get `1/lobes` of the motion. Keep an animated value in the same units as the thing it moves |
| A hard vertical line down the front | Something that is not periodic in azimuth: non-integer `lobes`, or noise sampled as `noise(azimuth)` |
| Stars or texture clump at the top | Points scattered uniformly in elevation. Use `arcsin(uniform)` |
| Low ribbons vanish behind the audience | Normal: behind, the picture only starts about 36° up. Keep the subject in front |